In [1]:
#This file shows early morning and evening spikes the CNT (Hours around work/school start and finish)
import pandas as pd
import plotly.express as px

bikes = pd.read_csv("hour.csv")
holidays = pd.read_csv("us_federal_holidays_2011_2012.csv", encoding="utf-8-sig")

# Convert to actual dates
bikes["dteday"] = pd.to_datetime(bikes["dteday"])
holidays["date"] = pd.to_datetime(holidays["date"], format="%d/%m/%Y")

# Drop the duplicates
holidays = (
    holidays[["date", "holiday"]]
    .rename(columns={"holiday": "holiday_name"})
    .drop_duplicates(subset=["date"])
)

# Join statment
bikes = bikes.merge(
    holidays,
    left_on="dteday",
    right_on="date",
    how="left",
    validate="many_to_one"
)

bikes["day_type"] = bikes["holiday_name"].notna().map({
    True: "Holiday",
    False: "Non-holiday"
})

daily = bikes.groupby(
    ["dteday", "day_type", "holiday_name"],
    dropna=False
).agg(
    daily_rentals=("cnt", "sum")
).reset_index()


# print(daily.head())
# print(daily["day_type"].value_counts())

# ------- We should put graphs after here -------

def classify(row):
    if pd.notna(row["holiday_name"]):
        return "Holiday"
    return "Weekend" if row["weekday"] in (0, 6) else "Weekday"

bikes["day_category"] = bikes.apply(classify, axis=1)

hourly = (
    bikes.groupby(["day_category", "hr"], as_index=False)
    .agg(avg_rentals=("cnt", "mean"))
)

fig1 = px.line(
    hourly,
    x="hr",
    y="avg_rentals",
    color="day_category",
    markers=True,
    title="Average Bike Rentals by Hour of Day",
    labels={"hr": "Hour of day", "avg_rentals": "Average rentals per hour",
            "day_category": "Day type"},
    color_discrete_map={"Weekday": "#1f77b4", "Weekend": "#2ca02c", "Holiday": "#d62728"},
)
fig1.update_xaxes(dtick=2)
fig1.show()